# MobileNetV2 op CIFAR-10 · geheugen en trainingsduur

**Any-Precision met W/A 32, 8, 4, 2 en 1 · K=3 · batch 128**

Dit notebook presenteert de korte haalbaarheidsproef. Het maakt geen verbinding met een GPU, start geen training en wijzigt geen checkpoint. Voer eerst de runner uit en daarna **Run All** in dit notebook.

De kernvragen zijn: passen volledige trainingsupdates in het GPU-geheugen, hoe lang duren ze met echte data en welke trainingsduur volgt daaruit? De voorspelde aantallen epochs zijn **ramingen**, geen uitgevoerde trainingen.


## 1. Opzet en herkomst

De proef gebruikt MobileNetV2 met breedtefactor 1 en stride 1 in de eerste convolutie. De centrale 3×3-convoluties hebben acht ingangskanalen per groep (`groups = c / 8`). Dit zijn convolutiegroepen, geen MX-blokken.

Elke trainingsstap omvat vijf forward/backward-passes met recursieve distillatie en één Adam-update. De mastergewichten, rekenopslag, eerste convolutie en laatste classifier blijven FP32. De lagere W/A-modi simuleren quantisatie op de vastgelegde interne quantisatiepunten. De lineaire bottleneck en residual-optelling krijgen geen extra clamp. Er is nog geen MX-quantisatie.

Architectuur, K=3 en batch 128 zijn vastgelegde reconstructiekeuzes. Het langere paperrecept is Adam met learning rate 0,001 en zonder weight decay, 400 epochs en learning-rateverlagingen op 150, 250 en 350. De exacte oorspronkelijke MobileNetV2-implementatie ontbreekt. Zie [het protocol](PROTOCOL.md) voor alle aannames.


In [ ]:
from pathlib import Path
import importlib.util
import sys

# Standaard: nieuwste rapport van deze proef, inclusief een eventueel FAIL-rapport.
# Kies zo nodig expliciet de map met run_report.json, ook na uitpakken van een feedbackbundel.
RUN_DIR = None
# Voorbeeld:
# RUN_DIR = Path(r"C:\Users\renzo\master-thesis-mx\mx_runs\anyprecision_cifar10_mobilenetv2_k3_timing\20261030_..._run")
PROJECT_ROOT = None
# PROJECT_ROOT = Path(r"C:\Users\renzo\master-thesis-mx")

package_name = "anyprecision_cifar10_mobilenetv2_k3_timing_v1"
origins = [Path.cwd(), *list(Path.cwd().parents)[:5]]
if PROJECT_ROOT is not None:
    origins.insert(0, Path(PROJECT_ROOT))
if RUN_DIR is not None:
    origins.insert(0, Path(RUN_DIR))
paths = []
for base in origins:
    paths.extend([base / "notebook_support.py",
                  base / "runner_source" / "notebook_support.py",
                  base / "software_checks" / package_name / "notebook_support.py",
                  base / package_name / "notebook_support.py"])
support_path = next((p for p in paths if p.is_file()), None)
if support_path is None:
    raise FileNotFoundError("notebook_support.py niet gevonden. Open het notebook vanuit de uitgepakte pakketmap, of vul PROJECT_ROOT in.")
spec = importlib.util.spec_from_file_location("mobilenet_k3_timing_notebook_support", support_path)
ns = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = ns
spec.loader.exec_module(ns)

import matplotlib.pyplot as plt
from IPython.display import display
plt.rcParams.update({"font.size": 10, "axes.titlesize": 13, "axes.titleweight": "bold",
                     "axes.labelcolor": "#263D4B", "text.color": "#263D4B",
                     "figure.facecolor": "white", "axes.facecolor": "white"})


In [ ]:
context = ns.load_run(RUN_DIR, PROJECT_ROOT)
display(ns.overview_table(context))
for note in ns.status_notes(context):
    print("•", note)


## 2. Controle van de groepsindeling

Hier is `c` het aantal ingangskanalen van de **centrale 3×3-convolutie**, na de eventuele expansie van het MobileNetV2-blok. Het aantal groepen kan dus per laag veranderen; het aantal ingangskanalen per groep blijft acht.


In [ ]:
groups = ns.group_table(context)
if groups.empty:
    print("Nog geen modelsamenvatting beschikbaar; controleer de status van de run.")
else:
    display(groups)


## 3. Duur van een volledige trainingsupdate

De figuur toont alle gemeten updates, inclusief de opwarming. Voor de proef met echte data tellen de eerste tien stappen niet mee in de stabiele tijdschatting. Voor de synthetische controle wordt één opwarmupdate gevolgd door drie meetupdates.

De totale wandtijd omvat datalevering, overdracht en berekening binnen het gemeten stapinterval. De berekening bevat alle vijf precisies, backward en Adam. De vijf passes zijn dus al in de stapduur verwerkt.


In [ ]:
figure = ns.plot_step_times(context)
if figure is None:
    print("Geen stapmetingen beschikbaar.")
else:
    plt.show()

timings = ns.timing_table(context)
if not timings.empty:
    display(timings.round(4))


## 4. GPU-geheugen

**Gealloceerd** en **gereserveerd** mogen niet worden opgeteld: gealloceerd geheugen is onderdeel van het gereserveerde allocatorgeheugen. De fysieke capaciteit is een bovengrens, geen volledig vrije capaciteit.

De proef begrenst de PyTorch-allocator vooraf op maximaal 85% van de fysieke capaciteit én maximaal 90% van het aanvankelijk vrije geheugen. Andere GPU-gebruikers en drivergeheugen vallen niet volledig onder die allocator. Dit is geen directe meting van Windows' gedeeld GPU-geheugen of van fysieke geheugenresidentie.


In [ ]:
memory = ns.memory_table(context)
if not memory.empty:
    display(memory.round(3))
figure = ns.plot_memory(context)
if figure is None:
    print("Geen GPU-geheugenmetingen beschikbaar.")
else:
    plt.show()


## 5. Verwachte duur van een langere training

De extrapolatie gaat uit van dezelfde batch, dezelfde vijf precisies en hetzelfde backendbeleid. Een volledige CIFAR-10-epoch omvat 391 trainingsbatches en 79 evaluatiebatches. De kleinere laatste batches worden conservatief als volle batches gerekend.

Een complete raming uit de dataproef omvat training, evaluatie op alle 10.000 testbeelden en checkpointopslag. Een synthetische meting geeft uitsluitend een indicatie voor de rekencomponent; ontbrekende datalevering en evaluatie worden daarbij niet als nul uur totale kosten geïnterpreteerd.

Een laag/hoog scenario op basis van de waargenomen stappercentielen is **geen betrouwbaarheidsinterval**. Opstart, eenmalige datasetvoorbereiding en eventuele latere wijzigingen van temperatuur, GPU-vermogen of andere systeembelasting kunnen de werkelijke looptijd veranderen. De raming sluit extra rapportserialisatie buiten de gemeten stapintervallen uit.


In [ ]:
forecast = ns.forecast_table(context)
if forecast.empty:
    print("Nog geen bruikbare tijdsprojectie beschikbaar.")
else:
    display(forecast.round(3))
    figure = ns.plot_forecast(context)
    if figure is not None:
        plt.show()

summary = context["worker"].get("timing_summary", {})
for note in summary.get("notes", []):
    print("•", note)


## 6. Vroege loss en accuracy · alleen ter diagnose

De dataproef verwerkt 100 batches, dus nog geen volledige epoch. De hier getoonde trainingmetrics worden over de korte proef samengevat; de evaluatie gebruikt 1.000 testbeelden met 100 voorbeelden per klasse. Deze waarden voorspellen niet de eindaccuracy en horen niet in een vergelijking met volledig getrainde paper- of MX-modellen.

De hoogste precisie gebruikt harde cross-entropy, terwijl de lagere precisies zachte distillatiedoelen gebruiken. De losswaarden zijn daarom geen gelijksoortige rangschikking van modelkwaliteit. Bij synthetische beelden worden geen accuracyresultaten gepresenteerd.


In [ ]:
for phase, title in [("training", "Training · gedeeltelijke diagnose"),
                     ("validation", "Evaluatie · diagnostische steekproef")]:
    table = ns.precision_table(context, phase)
    if not table.empty:
        print(title)
        display(table.round(3))
if context["worker"].get("kind") != "real":
    print("Synthetische controle: accuracy wordt bewust niet als onderzoeksresultaat getoond.")


## 7. Wat kunnen we hiermee beslissen?

- **Uitvoerbaarheid:** alleen een geslaagde controle bevestigt dat de gemeten route met de gekozen batch kon worden uitgevoerd. Een timeout bewijst geen OOM.
- **Tijdsbudget:** gebruik bij voorkeur de raming uit echte CIFAR-10-data. Vergelijk de verwachte duur van vijf en 400 epochs met het beschikbare onderzoeksbudget.
- **Vervolg:** een eventuele vijf-epochpilot of volledige training is een afzonderlijke stap en begint opnieuw vanaf nul.
- **Accuracygevoeligheid:** die blijft voor een langere, gecontroleerde training te onderzoeken. Deze tijdproef selecteert geen optimale groepsindeling en bewijst niet dat MobileNetV2 dezelfde precisiekloof als de paper heeft.

Deel de `feedback_bundle.zip` uit de resultaatmap voor de analyse. Dataset en diagnostische checkpoint blijven lokaal.
